In [ ]:
# ===== Cell 1: Setup, load all files, sanity check =====
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

def find_file(*names):
    for n in names:
        hits = glob.glob(f"/content/**/{glob.escape(n)}", recursive=True)
        if hits: return hits[0]
    raise FileNotFoundError(f"none of {names} found")

FILES = {
    'train':       find_file("train.csv", "train 1.csv"),
    'test':        find_file("test.csv", "test 1.csv"),
    'Llama_run1':  find_file("full_llama.csv", "full_llama 1.csv", "llama_run1.csv"),
    'Llama_run2':  find_file("llama_run2.csv"),
    'Llama_run3':  find_file("llama_run3.csv"),
    'Falcon_run1': find_file("falcon_full.csv", "falcon_full 1.csv", "falcon_run1.csv"),
    'Falcon_run2': find_file("falcon_run2.csv"),
    'Falcon_run3': find_file("falcon_run3.csv"),
    'Qwen_run1':   find_file("qwen_full.csv", "qwen_full 1.csv", "qwen_run1.csv"),
    'Qwen_run2':   find_file("qwen_run2.csv"),
    'Qwen_run3':   find_file("qwen_run3.csv"),
}
# CTGAN: full schema, 300 epochs (default), 3 seeds
for s in (42, 45, 50):
    hits = sorted(glob.glob(f"/content/**/ctgan_full_e300_s{s}*.csv", recursive=True))
    if not hits: raise FileNotFoundError(f"ctgan_full_e300_s{s}")
    FILES[f'CTGAN_s{s}'] = hits[0]

print("Resolved:")
for k, v in FILES.items(): print(f"  {k:12s} -> {v}")

train = pd.read_csv(FILES['train'])
test  = pd.read_csv(FILES['test'])
syn_raw = {k: pd.read_csv(v) for k, v in FILES.items() if k not in ('train', 'test')}

print("\ntrain:", train.shape, train['is_attack'].value_counts().to_dict())
print("test: ", test.shape,  test['is_attack'].value_counts().to_dict())
for k, df in syn_raw.items():
    miss = set(train.columns) - set(df.columns)
    print(f"{k:12s} {df.shape}  missing cols: {sorted(miss) if miss else 0}")

In [ ]:
# ===== Cell 2: Same prep/models as before + combined sets (LLMs and CTGAN) =====
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

test = test[train.columns]
attack_train = train[train['is_attack'] == True].drop(columns=['is_attack']).reset_index(drop=True)
benign_train = train[train['is_attack'] == False].reset_index(drop=True)
y_test = test['is_attack'].astype(int).values
assert len(attack_train) == 3116 and len(benign_train) == 404

ID_COLS = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b']
CONTEXT_PREFIX = ('source_', 'target_service_', 'target_ip_', 'flow_share_of_')

def load_syn_df(df):
    s = df.reindex(columns=train.columns).drop(columns=['is_attack'])
    for c in s.columns:
        if c in ID_COLS: continue
        if train[c].dtype == bool:
            s[c] = s[c].astype(str).str.strip().str.lower().map({'true':1,'false':0,'1':1,'0':0,'1.0':1,'0.0':0})
        else:
            s[c] = pd.to_numeric(s[c], errors='coerce')
    return s

def prep_full(df):
    X = df.drop(columns=[c for c in ID_COLS + ['is_attack'] if c in df.columns]).copy()
    for c in X.select_dtypes('bool').columns: X[c] = X[c].astype(int)
    return X

def prep_hard(df):
    X = prep_full(df)
    return X.drop(columns=[c for c in X.columns if c.startswith(CONTEXT_PREFIX)])

def make_models():
    return {
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'LogisticReg':  LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED),
        'SVM':          LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED),
        'MLP':          MLPClassifier(hidden_layer_sizes=(100,50), activation='relu', solver='adam',
                                      early_stopping=True, max_iter=500, random_state=SEED),
    }
MODELS = list(make_models().keys())

def fit_predict(attack_rows, prep_fn=prep_hard):
    a = attack_rows.copy(); a['is_attack'] = True
    tr  = pd.concat([a, benign_train], ignore_index=True)
    Xtr, ytr = prep_fn(tr), tr['is_attack'].astype(int)
    Xte = prep_fn(test)[Xtr.columns]
    preds = {}
    for name, m in make_models().items():
        pipe = make_pipeline(SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True),
                             StandardScaler(), m).fit(Xtr, ytr)
        preds[name] = pipe.predict(Xte).astype(int)
    return preds

def score(yp, generator, model):
    tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
    return {'Generator': generator, 'Model': model,
            'Acc': accuracy_score(y_test, yp), 'Prec': precision_score(y_test, yp, zero_division=0),
            'Rec': recall_score(y_test, yp), 'F1': f1_score(y_test, yp),
            'Benign_FPR': fp / (fp + tn), 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

# --- Sets ---
SYN = {k: load_syn_df(df) for k, df in syn_raw.items()}
LLM_NAMES = ['Llama', 'Falcon', 'Qwen']
for g in LLM_NAMES:
    SYN[f'{g}_combined'] = pd.concat([SYN[f'{g}_run{i}'] for i in (1, 2, 3)], ignore_index=True)
SYN['CTGAN_combined'] = pd.concat([SYN[f'CTGAN_s{s}'] for s in (42, 45, 50)], ignore_index=True)

for g, s in SYN.items():
    print(f'{g:16s} rows={len(s):5d}  dups={s.duplicated().sum()}')
print('hard features:', prep_hard(test).shape[1])

In [ ]:
# ===== Cell 3: Train all sets, ensembles, final table (HARD features) =====
SETS = {'Real (TRTR)': attack_train, **SYN}
PREDS = {}
for g, s in SETS.items():
    print('training', g); PREDS[g] = fit_predict(s)

# Within-LLM vote (run1/2/3 of same LLM) -> one line in text only
for g in LLM_NAMES:
    PREDS[f'{g}_ensemble'] = {m: (sum(PREDS[f'{g}_run{r}'][m] for r in (1, 2, 3)) >= 2).astype(int)
                              for m in MODELS}

# Cross-LLM vote: same classifier, 3 LLMs vote
PREDS['XLLM_vote_combined'] = {m: (sum(PREDS[f'{g}_combined'][m] for g in LLM_NAMES) >= 2).astype(int)
                               for m in MODELS}                                   # MAIN (2 of 3)
PREDS['XLLM_vote_9runs'] = {m: (sum(PREDS[f'{g}_run{r}'][m] for g in LLM_NAMES for r in (1, 2, 3)) >= 5).astype(int)
                            for m in MODELS}                                      # 5 of 9

# Pooled control: 3,300 from each LLM -> 9,900 rows
pooled = pd.concat([SYN[f'{g}_combined'].sample(3300, random_state=42) for g in LLM_NAMES], ignore_index=True)
print('training XLLM_pooled'); PREDS['XLLM_pooled'] = fit_predict(pooled)

# --- Score everything ---
res = pd.DataFrame([score(PREDS[g][m], g, m) for g in PREDS for m in MODELS])
res.to_csv('tstr_hard_final_ensemble.csv', index=False)
F1 = res.pivot(index='Generator', columns='Model', values='F1')[MODELS]
F1['Mean'] = F1[MODELS].mean(axis=1)
FPR = res.groupby('Generator')['Benign_FPR'].mean()

# --- Sanity: must match earlier results ---
expected = {'Real (TRTR)': .994,
            'Llama_run1': .782, 'Llama_run2': .733, 'Llama_run3': .625, 'Llama_combined': .728,
            'Falcon_run1': .827, 'Falcon_run2': .849, 'Falcon_run3': .765, 'Falcon_combined': .888,
            'Qwen_run1': .892, 'Qwen_run2': .904, 'Qwen_run3': .916, 'Qwen_combined': .910,
            'CTGAN_s42': .969, 'CTGAN_s45': .966, 'CTGAN_s50': .966}
bad = {k: round(F1.loc[k, 'Mean'], 3) for k, v in expected.items() if abs(F1.loc[k, 'Mean'] - v) >= .0015}
print('\nMatches earlier results:', not bad, '' if not bad else f'-> mismatches: {bad}')

# --- Final table ---
def ms(names):   # mean ± std across runs/seeds
    sub = F1.loc[names, MODELS + ['Mean']]
    out = {c: f"{sub[c].mean():.3f} ± {sub[c].std():.3f}" for c in MODELS + ['Mean']}
    out['FPR'] = f"{FPR[names].mean():.3f}"
    return out

def one(name):
    out = {c: f"{F1.loc[name, c]:.3f}" for c in MODELS + ['Mean']}
    out['FPR'] = f"{FPR[name]:.3f}"
    return out

rows = {'Real (TRTR)': one('Real (TRTR)'),
        'CTGAN individual (3 seeds)': ms(['CTGAN_s42', 'CTGAN_s45', 'CTGAN_s50']),
        'CTGAN combined (9,900)': one('CTGAN_combined')}
for g in LLM_NAMES:
    rows[f'{g} individual (3 runs)'] = ms([f'{g}_run{r}' for r in (1, 2, 3)])
    rows[f'{g} combined (9,900)'] = one(f'{g}_combined')
rows['Cross-LLM vote, combined (2 of 3)'] = one('XLLM_vote_combined')
rows['Cross-LLM vote, 9 runs (5 of 9)'] = one('XLLM_vote_9runs')
rows['Cross-LLM pooled (9,900)'] = one('XLLM_pooled')

final = pd.DataFrame(rows).T[MODELS + ['Mean', 'FPR']]
final.to_csv('final_table_f1.csv')
print('\n===== FINAL TABLE: TSTR-HARD F1 =====')
print(final.to_string())

print('\n===== Within-LLM vote (text only) =====')
print(F1.loc[[f'{g}_ensemble' for g in LLM_NAMES], 'Mean'].round(3).to_string())

In [ ]:
# ===== Share table: RF, LR, SVM only =====
M3 = ['RandomForest', 'LogisticReg', 'SVM']
F3 = F1[M3].copy(); F3['Mean'] = F3[M3].mean(axis=1)
FPR3 = res[res['Model'].isin(M3)].groupby('Generator')['Benign_FPR'].mean()

def ms3(names):   # mean ± std across runs/seeds
    sub = F3.loc[names]
    out = {c: f"{sub[c].mean():.3f} ± {sub[c].std():.3f}" for c in M3 + ['Mean']}
    out['FPR'] = f"{FPR3[names].mean():.3f}"
    return out

def one3(name):
    out = {c: f"{F3.loc[name, c]:.3f}" for c in M3 + ['Mean']}
    out['FPR'] = f"{FPR3[name]:.3f}"
    return out

rows = {'CTGAN individual (3 seeds)': ms3(['CTGAN_s42', 'CTGAN_s45', 'CTGAN_s50']),
        'CTGAN combined (9,900)':     one3('CTGAN_combined')}
for g in LLM_NAMES:
    rows[f'{g} individual (3 runs)'] = ms3([f'{g}_run{r}' for r in (1, 2, 3)])
    rows[f'{g} combined (9,900)']    = one3(f'{g}_combined')
rows['Cross-LLM ensemble (vote of combined, 2 of 3)'] = one3('XLLM_vote_combined')

share = pd.DataFrame(rows).T[M3 + ['Mean', 'FPR']]
share.columns = ['RF', 'LR', 'SVM', 'Mean F1 (RF/LR/SVM)', 'Benign FPR']
share.index.name = 'Training data'

print('TSTR-Hard F1 (85 features), MLP not shown\n')
print(share.to_markdown())
share.to_csv('share_table_rf_lr_svm.csv')
from google.colab import files; files.download('share_table_rf_lr_svm.csv')

In [ ]:
# ===== Fidelity: KS, JSD, MMD² vs real TEST attacks (HARD) — share rows only =====
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics.pairwise import euclidean_distances

def ks_jsd_all(R, X, bins=20):
    ks, js = [], []
    for c in R.columns:
        r, x = R[c].values, X[c].values
        if np.ptp(np.r_[r, x]) == 0:
            ks.append(0.0); js.append(0.0); continue
        ks.append(ks_2samp(r, x).statistic)
        edges = np.unique(np.quantile(np.r_[r, x], np.linspace(0, 1, bins + 1)))
        if len(edges) < 2: edges = np.array([edges[0] - 0.5, edges[0] + 0.5])
        pr, _ = np.histogram(r, edges); px, _ = np.histogram(x, edges)
        js.append(jensenshannon(pr + 1e-12, px + 1e-12, base=2) ** 2)
    return np.mean(ks), np.mean(js)

def subs(A, n, seed): return A[np.random.default_rng(seed).choice(len(A), min(n, len(A)), replace=False)]

test_att = test[test['is_attack'] == True].drop(columns=['is_attack'])
R  = prep_hard(test_att).fillna(0).astype(float)
qt = QuantileTransformer(n_quantiles=1000, output_distribution='uniform', random_state=SEED).fit(R)
Rz = qt.transform(R)
d0 = euclidean_distances(subs(Rz, 1000, 0), squared=True)
gams = [1.0 / (np.median(d0[d0 > 0]) * f) for f in (0.1, 1, 10)]
Kf = lambda P, Q: sum(np.exp(-g * euclidean_distances(P, Q, squared=True)) for g in gams) / len(gams)

def mmd2(Xz, n=1000):
    A, B = subs(Rz, n, 1), subs(Xz, n, 2); m, k = len(A), len(B)
    Kxx, Kyy, Kxy = Kf(A, A), Kf(B, B), Kf(A, B)
    return (Kxx.sum() - np.trace(Kxx)) / (m*(m-1)) + (Kyy.sum() - np.trace(Kyy)) / (k*(k-1)) - 2 * Kxy.mean()

# --- Score each dataset needed for the share rows ---
NEED = ['CTGAN_s42', 'CTGAN_s45', 'CTGAN_s50', 'CTGAN_combined'] + \
       [f'{g}_{r}' for g in LLM_NAMES for r in ('run1', 'run2', 'run3', 'combined')]
FSETS = {'Real-train (ref)': attack_train, **{k: SYN[k] for k in NEED}}
fr = {}
for g, df in FSETS.items():
    X = prep_hard(df)[R.columns].fillna(0).astype(float)
    ks, js = ks_jsd_all(R, X)
    fr[g] = {'KS': ks, 'JSD': js, 'MMD2': mmd2(qt.transform(X))}
fr = pd.DataFrame(fr).T
fr.to_csv('fidelity_hard_per_set.csv')

# --- Sanity: run-3 + Real-train must match earlier ---
chk = {'Real-train (ref)': (.0089, .0010, -.0003), 'Llama_run3': (.2852, .2677, .2315),
       'Falcon_run3': (.2611, .2525, .1962), 'Qwen_run3': (.3364, .2930, .2256)}
bad = {k: fr.loc[k].round(4).tolist() for k, v in chk.items() if max(abs(fr.loc[k].values - np.array(v))) > .0006}
print('Matches earlier fidelity:', not bad, '' if not bad else f'-> {bad}')

# --- Share table ---
M = ['KS', 'JSD', 'MMD2']
def ms_f(names): return {c: f"{fr.loc[names, c].mean():.3f} ± {fr.loc[names, c].std():.3f}" for c in M}
def one_f(n):    return {c: f"{fr.loc[n, c]:.3f}" for c in M}

rows = {'Real-train (reference)':     one_f('Real-train (ref)'),
        'CTGAN individual (3 seeds)': ms_f(['CTGAN_s42', 'CTGAN_s45', 'CTGAN_s50']),
        'CTGAN combined (9,900)':     one_f('CTGAN_combined')}
for g in LLM_NAMES:
    rows[f'{g} individual (3 runs)'] = ms_f([f'{g}_run{r}' for r in (1, 2, 3)])
    rows[f'{g} combined (9,900)']    = one_f(f'{g}_combined')
rows['Cross-LLM ensemble (vote of combined, 2 of 3)'] = {c: '—' for c in M}

fid_share = pd.DataFrame(rows).T[M]
fid_share.columns = ['KS', 'JSD', 'MMD²']
fid_share.index.name = 'Training data'
print('\nFidelity vs 1,346 real test attacks, HARD (85 features), lower = better\n')
print(fid_share.to_markdown())
fid_share.to_csv('share_table_fidelity.csv')
from google.colab import files; files.download('share_table_fidelity.csv')

In [ ]:
# ===== Cell F1: Fusion pre-checks (alignment, types, rules.json) =====
import json

# --- 1) Row alignment: same generation slot per row index? ---
print('=== Row alignment (timeframe_id per row index) ===')
for r in (1, 2, 3):
    tf = {g: syn_raw[f'{g}_run{r}']['timeframe_id'].astype(str).values for g in LLM_NAMES}
    same = (tf['Llama'] == tf['Falcon']) & (tf['Falcon'] == tf['Qwen'])
    print(f'run{r}: {same.mean():.1%} of 3,300 rows aligned')

# --- 2) Auto-type features with the 4 rules (schema from benign train only) ---
feat_cols = [c for c in train.columns if c not in ID_COLS + ['is_attack']]
TYPES = {}
for c in feat_cols:
    if train[c].dtype == bool:
        TYPES[c] = 'binary'
    elif train[c].dtype == object:
        TYPES[c] = 'nominal'
    else:
        v = pd.to_numeric(benign_train[c], errors='coerce').dropna()
        if len(v) and set(v.unique()) <= {0, 1}: TYPES[c] = 'binary'
        elif len(v) and (v % 1 == 0).all():    TYPES[c] = 'integer'
        else:                                  TYPES[c] = 'continuous'
print('\n=== Feature types ===')
print(pd.Series(TYPES).value_counts().to_string())
print('nominal cols:', [c for c, t in TYPES.items() if t == 'nominal'])

# --- 3) rules.json structure (for recomputing derived fields after fusion) ---
hits = glob.glob('/content/**/rules.json', recursive=True)
if hits:
    rules = json.load(open(hits[0]))
    print('\n=== rules.json ===')
    print('top-level keys:', list(rules.keys()) if isinstance(rules, dict) else type(rules))
    for k, v in (rules.items() if isinstance(rules, dict) else []):
        print(f'\n[{k}] type={type(v).__name__} len={len(v) if hasattr(v, "__len__") else "-"}')
        print('  sample:', str(list(v.items())[:3] if isinstance(v, dict) else v[:3])[:400])
else:
    print('\n rules.json not found, upload it')

In [ ]:
# ===== Cell F2: Feature-level fusion of Llama/Falcon/Qwen (rules.json roles) =====
import json
rules = json.load(open(sorted(glob.glob('/content/**/rules*.json', recursive=True))[0]))

ID_FUSE = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b']
NOMINAL = ['proto']
ROLE = {}
for c in SYN['Falcon_run1'].columns:
    if c in ID_FUSE:                      ROLE[c] = 'id'
    elif c in rules['derived']:           ROLE[c] = 'derived'
    elif c in rules['pinned']:            ROLE[c] = 'pinned'
    elif c in rules['always_empty']:      ROLE[c] = 'empty'
    elif c in NOMINAL:                    ROLE[c] = 'nominal'
    elif rules['dtypes'].get(c) == 'int': ROLE[c] = 'integer'
    else:                                 ROLE[c] = 'continuous'
print('=== Column roles ===\n', pd.Series(ROLE).value_counts().to_string())
FUSE_NUM = [c for c, t in ROLE.items() if t in ('integer', 'continuous')]
DER = [c for c in rules['derived'] if c in SYN['Falcon_run1'].columns]

# ---------- the 4 fusion rules ----------
def fuse_numeric(V, integer=False):           # V: (n, 3) with NaN = empty
    out = np.full(len(V), np.nan); n_nan = np.isnan(V).sum(1)
    m1 = n_nan == 1                            # 2 present -> average them
    out[m1] = np.nanmean(V[m1], axis=1)
    m0 = n_nan == 0                            # 3 present -> average of the two closest
    A = np.sort(V[m0], axis=1); g1, g2 = A[:, 1] - A[:, 0], A[:, 2] - A[:, 1]
    out[m0] = np.where(g1 < g2, (A[:, 0] + A[:, 1]) / 2,
              np.where(g2 < g1, (A[:, 1] + A[:, 2]) / 2, A[:, 1]))   # equal gaps -> middle
    return np.floor(out) if integer else out   # 2+ empty -> stays empty

def fuse_nominal(V, rng):                     # majority; all differ -> seeded random pick
    out, n_diff = [], 0
    for a, b, c in V:
        if a == b or a == c: out.append(a)
        elif b == c:         out.append(b)
        else:                out.append([a, b, c][rng.integers(3)]); n_diff += 1
    return np.array(out), n_diff

# ---------- rules.json post-processing ----------
def recompute_derived(d):
    d = d.copy()
    for col, f in rules['derived'].items():
        if col in d.columns:
            d[col] = pd.Series(d.eval(f, engine='python'), index=d.index).astype(float)\
                       .replace([np.inf, -np.inf], np.nan)
    return d

def apply_empty_rules(d):
    for er in rules['empty_rules']:
        for cond in er['must_be_empty_if']:
            try: m = d.eval(cond, engine='python').astype(bool)
            except Exception: continue          # skips non-code rules like "INT missing"
            d.loc[m, [c for c in er['columns'] if c in d.columns]] = np.nan
    return d

def derived_agreement(d):                     # % of derived cells already matching formulas
    r = apply_empty_rules(recompute_derived(d))
    a, b = d[DER].astype(float).values, r[DER].astype(float).values
    return (np.isclose(a, b, rtol=1e-3, atol=1e-6) | (np.isnan(a) & np.isnan(b))).mean()

# ---------- fuse one run ----------
def fuse_run(r, seed=42):
    rng = np.random.default_rng(seed)
    srcs = [SYN[f'{g}_run{r}'].reset_index(drop=True) for g in LLM_NAMES]
    out = srcs[1].copy()                                   # IDs copied (dropped by ML anyway)
    rep = {}
    for c in [c for c, t in ROLE.items() if t == 'nominal']:
        out[c], rep[f'{c}: all 3 differ'] = fuse_nominal(np.stack([s[c].values for s in srcs], 1), rng)
    n_empty = 0
    for c in FUSE_NUM:
        V = np.stack([pd.to_numeric(s[c], errors='coerce').values.astype(float) for s in srcs], 1)
        if c == 'duration': Vdur = V
        n_empty += int((np.isnan(V).sum(1) >= 2).sum())
        out[c] = fuse_numeric(V, integer=(ROLE[c] == 'integer'))
    rep['cells empty by 2-of-3 vote'] = n_empty

    for c, v in rules['pinned'].items():
        if c in out.columns and c not in ID_FUSE: out[c] = v
    for c in rules['always_empty']:
        if c in out.columns: out[c] = np.nan

    # constraint: pkts_fwd == 1 <=> duration == 0
    p1 = out['pkts_fwd'] == 1
    rep['duration -> 0 (pkts_fwd==1)'] = int((p1 & (out['duration'] != 0)).sum())
    out.loc[p1, 'duration'] = 0.0
    bad = ((out['pkts_fwd'] > 1) & (out['duration'] == 0)).values
    alt = np.nanmean(np.where(Vdur > 0, Vdur, np.nan), axis=1)
    fix = bad & ~np.isnan(alt); out.loc[fix, 'duration'] = alt[fix]
    rep['duration filled (pkts_fwd>1, dur==0)'] = int(fix.sum())
    rep['still pkts_fwd>1 & dur==0'] = int((bad & ~fix).sum())

    # constraint: flag counts <= pkts_fwd
    n_clip = 0
    for f in ('syn', 'ack', 'fin', 'rst', 'psh', 'urg'):
        c = f'{f}_count_fwd'; over = out[c] > out['pkts_fwd']
        n_clip += int(over.sum()); out.loc[over, c] = out.loc[over, 'pkts_fwd']
    rep['flag counts clipped to pkts_fwd'] = n_clip

    out = recompute_derived(out)

    # constraint: min <= mean <= max
    for lo, mid, hi in [('pkt_size_min', 'mean_pkt_size', 'pkt_size_max'),
                        ('iat_min_int_ns', 'mean_iat_int_ns', 'iat_max_int_ns'),
                        ('latency_min', 'mean_hop_latency', 'latency_max')]:
        rep[f'{lo}<={mid}<={hi} fixed'] = int(((out[lo] > out[mid]) | (out[mid] > out[hi])).sum())
        out[lo], out[hi] = np.fmin(out[lo], out[mid]), np.fmax(out[hi], out[mid])
    out['pkt_size_min'], out['pkt_size_max'] = np.floor(out['pkt_size_min']), np.ceil(out['pkt_size_max'])

    return apply_empty_rules(out), rep

# ---------- run ----------
reports = {}
for r in (1, 2, 3):
    SYN[f'Fused_run{r}'], reports[f'run{r}'] = fuse_run(r)
    SYN[f'Fused_run{r}'].to_csv(f'/content/fused_run{r}.csv', index=False)
SYN['Fused_combined'] = pd.concat([SYN[f'Fused_run{r}'] for r in (1, 2, 3)], ignore_index=True)

print('\n=== Fusion report (per run) ===')
print(pd.DataFrame(reports).to_string())

print('\n=== Derived-formula agreement (1.0 = fully consistent) ===')
for g in LLM_NAMES + ['Fused']:
    print(f'{g:7s}', ' '.join(f'run{r}={derived_agreement(SYN[f"{g}_run{r}"]):.3f}' for r in (1, 2, 3)))

for k in ('Fused_run1', 'Fused_run2', 'Fused_run3', 'Fused_combined'):
    print(f'{k:15s} rows={len(SYN[k])}  dups={SYN[k].duplicated().sum()}')

In [ ]:
# ===== Cell F3: TSTR + fidelity for fused data =====
FUSED = ['Fused_run1', 'Fused_run2', 'Fused_run3', 'Fused_combined']
for g in FUSED:
    print('training', g); PREDS[g] = fit_predict(SYN[g])

res = pd.concat([res[~res['Generator'].isin(FUSED)],
                 pd.DataFrame([score(PREDS[g][m], g, m) for g in FUSED for m in MODELS])], ignore_index=True)
F1 = res.pivot(index='Generator', columns='Model', values='F1')[MODELS]
F1['Mean'] = F1[MODELS].mean(axis=1)
F1['Mean_RF_LR_SVM'] = F1[['RandomForest', 'LogisticReg', 'SVM']].mean(axis=1)
FPR = res.groupby('Generator')['Benign_FPR'].mean()

# --- fidelity (same reference + settings as before) ---
for g in FUSED:
    X = prep_hard(SYN[g])[R.columns].fillna(0).astype(float)
    ks, js = ks_jsd_all(R, X)
    fr.loc[g] = {'KS': ks, 'JSD': js, 'MMD2': mmd2(qt.transform(X))}

# --- table ---
COLS = MODELS + ['Mean', 'Mean_RF_LR_SVM']
def ind(names):
    d = {c: f"{F1.loc[names, c].mean():.3f} ± {F1.loc[names, c].std():.3f}" for c in COLS}
    d['FPR'] = f"{FPR[names].mean():.3f}"
    d.update({c: f"{fr.loc[names, c].mean():.3f} ± {fr.loc[names, c].std():.3f}" for c in ['KS', 'JSD', 'MMD2']})
    return d
def one(n):
    d = {c: f"{F1.loc[n, c]:.3f}" for c in COLS}; d['FPR'] = f"{FPR[n]:.3f}"
    d.update({c: f"{fr.loc[n, c]:.3f}" for c in ['KS', 'JSD', 'MMD2']})
    return d

rows = {'CTGAN individual (3 seeds)': ind(['CTGAN_s42', 'CTGAN_s45', 'CTGAN_s50']),
        'CTGAN combined (9,900)':     one('CTGAN_combined')}
for g in LLM_NAMES:
    rows[f'{g} individual (3 runs)'] = ind([f'{g}_run{r}' for r in (1, 2, 3)])
    rows[f'{g} combined (9,900)']    = one(f'{g}_combined')
rows['FUSED individual (3 runs)'] = ind(['Fused_run1', 'Fused_run2', 'Fused_run3'])
rows['FUSED combined (9,900)']    = one('Fused_combined')

tab = pd.DataFrame(rows).T[COLS + ['FPR', 'KS', 'JSD', 'MMD2']]
tab = tab.rename(columns={'Mean': 'Mean F1 (4)', 'Mean_RF_LR_SVM': 'Mean F1 (RF/LR/SVM)', 'MMD2': 'MMD²'})
tab.to_csv('fused_results.csv')
print('\n===== TSTR-HARD F1 + Fidelity (lower = better for KS/JSD/MMD²) =====\n')
print(tab.to_markdown())

In [ ]:
# ===== Final results table =====
COLS = MODELS + ['Mean', 'Mean_RF_LR_SVM']
FID  = ['KS', 'JSD', 'MMD2']

def ind(names):   # mean ± std across runs/seeds
    d = {c: f"{F1.loc[names, c].mean():.3f} ± {F1.loc[names, c].std():.3f}" for c in COLS}
    d['FPR'] = f"{FPR[names].mean():.3f}"
    d.update({c: f"{fr.loc[names, c].mean():.3f} ± {fr.loc[names, c].std():.3f}" for c in FID})
    return d

def one(n, fid=None):
    d = {c: f"{F1.loc[n, c]:.3f}" for c in COLS}; d['FPR'] = f"{FPR[n]:.3f}"
    d.update({c: (f"{fr.loc[fid, c]:.3f}" if fid else '—') for c in FID})
    return d

rows = {'Real (TRTR)':                one('Real (TRTR)', fid='Real-train (ref)'),
        'CTGAN individual (3 seeds)': ind(['CTGAN_s42', 'CTGAN_s45', 'CTGAN_s50']),
        'CTGAN combined (9,900)':     one('CTGAN_combined', fid='CTGAN_combined')}
for g in LLM_NAMES:
    rows[f'{g} individual (3 runs)'] = ind([f'{g}_run{r}' for r in (1, 2, 3)])
    rows[f'{g} combined (9,900)']    = one(f'{g}_combined', fid=f'{g}_combined')
rows['Cross-LLM ensemble (vote, 2 of 3)'] = one('XLLM_vote_combined')            # no fidelity
rows['Fused individual (3 runs)']         = ind(['Fused_run1', 'Fused_run2', 'Fused_run3'])
rows['Fused combined (9,900)']            = one('Fused_combined', fid='Fused_combined')

final = pd.DataFrame(rows).T[COLS + ['FPR'] + FID]
final.columns = ['RF', 'LR', 'SVM', 'MLP', 'Mean F1 (4)', 'Mean F1 (RF/LR/SVM)',
                 'Benign FPR', 'KS', 'JSD', 'MMD²']
final.index.name = 'Training data'

print('TSTR-Hard F1 (85 features) + fidelity vs 1,346 real test attacks (lower = better)\n')
print(final.to_markdown())
final.to_csv('final_results_all.csv')
from google.colab import files; files.download('final_results_all.csv')

In [ ]:
################## fusion-medoid ####################################################

# ===== Cell M1: Row-level medoid selection + TSTR + fidelity =====
from sklearn.preprocessing import QuantileTransformer

DIST_COLS = [c for c in FUSE_NUM if not c.startswith(CONTEXT_PREFIX)]
pick_counts = {}
for r in (1, 2, 3):
    srcs = [SYN[f'{g}_run{r}'].reset_index(drop=True) for g in LLM_NAMES]
    Xs = [s[DIST_COLS].apply(pd.to_numeric, errors='coerce').fillna(0).values for s in srcs]
    qt_s = QuantileTransformer(n_quantiles=1000, random_state=SEED).fit(np.vstack(Xs))  # synthetic only
    L, F, Q = (qt_s.transform(x) for x in Xs)
    dLF, dLQ, dFQ = (np.linalg.norm(a - b, axis=1) for a, b in [(L, F), (L, Q), (F, Q)])

    tot = np.stack([dLF + dLQ, dLF + dFQ, dLQ + dFQ], 1)          # 0=Llama 1=Falcon 2=Qwen
    is_min = np.isclose(tot, tot.min(1, keepdims=True))
    ties = is_min.sum(1) > 1
    pick = np.argmin(tot, axis=1)
    pick[ties & is_min[:, 1]] = 1                                  # tie involving Falcon -> Falcon
    print(f'run{r} ties: {int(ties.sum())}')

    SYN[f'Medoid_run{r}'] = pd.concat([srcs[k].iloc[[i]] for i, k in enumerate(pick)], ignore_index=True)
    pick_counts[f'run{r}'] = {g: int((pick == k).sum()) for k, g in enumerate(LLM_NAMES)}
SYN['Medoid_combined'] = pd.concat([SYN[f'Medoid_run{r}'] for r in (1, 2, 3)], ignore_index=True)
print('\n=== Which LLM row was kept (per 3,300) ===\n', pd.DataFrame(pick_counts).to_string())

# --- TSTR + fidelity, same pipeline/test set ---
MED = ['Medoid_run1', 'Medoid_run2', 'Medoid_run3', 'Medoid_combined']
for g in MED:
    print('training', g); PREDS[g] = fit_predict(SYN[g])
    X = prep_hard(SYN[g])[R.columns].fillna(0).astype(float)
    ks, js = ks_jsd_all(R, X); fr.loc[g] = {'KS': ks, 'JSD': js, 'MMD2': mmd2(qt.transform(X))}

res = pd.concat([res[~res['Generator'].isin(MED)],
                 pd.DataFrame([score(PREDS[g][m], g, m) for g in MED for m in MODELS])], ignore_index=True)
F1 = res.pivot(index='Generator', columns='Model', values='F1')[MODELS]
F1['Mean'] = F1[MODELS].mean(axis=1)
F1['Mean_RF_LR_SVM'] = F1[['RandomForest', 'LogisticReg', 'SVM']].mean(axis=1)
FPR = res.groupby('Generator')['Benign_FPR'].mean()

# --- add to final table ---
rows['Medoid individual (3 runs)'] = ind(['Medoid_run1', 'Medoid_run2', 'Medoid_run3'])
rows['Medoid combined (9,900)']    = one('Medoid_combined', fid='Medoid_combined')
final = pd.DataFrame(rows).T[COLS + ['FPR'] + FID]
final.columns = ['RF', 'LR', 'SVM', 'MLP', 'Mean F1 (4)', 'Mean F1 (RF/LR/SVM)',
                 'Benign FPR', 'KS', 'JSD', 'MMD²']
final.index.name = 'Training data'
print('\nTSTR-Hard F1 (85 features) + fidelity vs 1,346 real test attacks (lower = better)\n')
print(final.to_markdown())
final.to_csv('final_results_all.csv')
from google.colab import files; files.download('final_results_all.csv')


In [ ]:
# ===== Per-column KS spread: what the averages hide (HARD, vs 1,346 real test attacks) =====
from scipy.stats import ks_2samp

def ks_per_col(X):
    return pd.Series({c: 0.0 if np.ptp(np.r_[R[c].values, X[c].values]) == 0
                      else ks_2samp(R[c].values, X[c].values).statistic for c in R.columns})

const_real = [c for c in R.columns if R[c].nunique() == 1]
nonconst   = [c for c in R.columns if c not in const_real]
print(f'{len(const_real)} of {R.shape[1]} columns are constant in real test attacks '
      f'-> {len(nonconst)} non-constant\n')

GROUPS = {'Real-train (reference)':     ['Real-train (ref)'],
          'CTGAN individual (3 seeds)': ['CTGAN_s42', 'CTGAN_s45', 'CTGAN_s50'],
          'CTGAN combined (9,900)':     ['CTGAN_combined']}
for g in LLM_NAMES:
    GROUPS[f'{g} individual (3 runs)'] = [f'{g}_run{r}' for r in (1, 2, 3)]
    GROUPS[f'{g} combined (9,900)']    = [f'{g}_combined']
for g in ('Fused', 'Medoid'):
    GROUPS[f'{g} individual (3 runs)'] = [f'{g}_run{r}' for r in (1, 2, 3)]
    GROUPS[f'{g} combined (9,900)']    = [f'{g}_combined']

# --- per-column KS for every dataset ---
need = sorted({n for v in GROUPS.values() for n in v})
src  = {'Real-train (ref)': attack_train, **SYN}
KSC  = pd.DataFrame({n: ks_per_col(prep_hard(src[n])[R.columns].fillna(0).astype(float)) for n in need})
KSC.to_csv('ks_per_column.csv')

def stats(n):
    k = KSC[n]
    return {'KS>0.5': (k > .5).sum(), 'KS>0.8': (k > .8).sum(), 'KS<0.05': (k < .05).sum(),
            'Mean KS (all 85)': k.mean(), f'Mean KS (non-const {len(nonconst)})': k[nonconst].mean(),
            'Max KS': k.max()}

rows_ks = {}
for label, names in GROUPS.items():
    S = pd.DataFrame([stats(n) for n in names])
    if len(names) == 1:
        rows_ks[label] = {c: (f"{int(S[c][0])}" if c.startswith('KS') else f"{S[c][0]:.3f}") for c in S}
    else:
        rows_ks[label] = {c: (f"{S[c].mean():.1f} ± {S[c].std():.1f}" if c.startswith('KS')
                              else f"{S[c].mean():.3f} ± {S[c].std():.3f}") for c in S}
spread = pd.DataFrame(rows_ks).T
spread.index.name = 'Training data'
print('===== Per-column KS spread (counts out of 85 columns) =====\n')
print(spread.to_markdown())
spread.to_csv('ks_spread.csv')

# --- worst 5 columns per combined set ---
print('\n===== Worst 5 columns (highest KS) =====')
for n in ['CTGAN_combined'] + [f'{g}_combined' for g in LLM_NAMES] + ['Fused_combined', 'Medoid_combined']:
    w = KSC[n].sort_values(ascending=False).head(5)
    print(f'{n:16s}', ', '.join(f'{c} ({v:.2f})' for c, v in w.items()))

from google.colab import files
files.download('ks_spread.csv'); files.download('ks_per_column.csv')

In [ ]:
#################### desc only + attack sandharva ################

## ===== Cell 1: Load new files + identical prep/models =====
import warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

D = "/content"
FILES = {
    'Llama_desc':  f"{D}/desc_only_llama.csv",
    'Falcon_desc': f"{D}/desc_only_falcon.csv",
    'Qwen_desc':   f"{D}/desc_only_qwen.csv",
    'Llama_B':     f"{D}/llama_B_run1.csv",
    'Falcon_B':    f"{D}/falcon_B_run1.csv",
    'Qwen_B':      f"{D}/qwen_B_run1.csv",
}
train = pd.read_csv(f"{D}/train.csv")
test  = pd.read_csv(f"{D}/test.csv")
syn_raw = {k: pd.read_csv(v) for k, v in FILES.items()}

print("train:", train.shape, train['is_attack'].value_counts().to_dict())
print("test: ", test.shape,  test['is_attack'].value_counts().to_dict())
for k, df in syn_raw.items():
    miss = set(train.columns) - set(df.columns)
    print(f"{k:12s} {df.shape}  missing cols: {sorted(miss) if miss else 0}")

# ---------- identical to reference notebook ----------
test = test[train.columns]
attack_train = train[train['is_attack'] == True].drop(columns=['is_attack']).reset_index(drop=True)
benign_train = train[train['is_attack'] == False].reset_index(drop=True)
y_test = test['is_attack'].astype(int).values
assert len(attack_train) == 3116 and len(benign_train) == 404

ID_COLS = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b']
CONTEXT_PREFIX = ('source_', 'target_service_', 'target_ip_', 'flow_share_of_')

def load_syn_df(df):
    s = df.reindex(columns=train.columns).drop(columns=['is_attack'])
    for c in s.columns:
        if c in ID_COLS: continue
        if train[c].dtype == bool:
            s[c] = s[c].astype(str).str.strip().str.lower().map({'true':1,'false':0,'1':1,'0':0,'1.0':1,'0.0':0})
        else:
            s[c] = pd.to_numeric(s[c], errors='coerce')
    return s

def prep_full(df):
    X = df.drop(columns=[c for c in ID_COLS + ['is_attack'] if c in df.columns]).copy()
    for c in X.select_dtypes('bool').columns: X[c] = X[c].astype(int)
    return X

def prep_hard(df):
    X = prep_full(df)
    return X.drop(columns=[c for c in X.columns if c.startswith(CONTEXT_PREFIX)])

def make_models():
    return {
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'LogisticReg':  LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED),
        'SVM':          LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED),
        'MLP':          MLPClassifier(hidden_layer_sizes=(100,50), activation='relu', solver='adam',
                                      early_stopping=True, max_iter=500, random_state=SEED),
    }
MODELS = list(make_models().keys())

def fit_predict(attack_rows, prep_fn=prep_hard):
    a = attack_rows.copy(); a['is_attack'] = True
    tr  = pd.concat([a, benign_train], ignore_index=True)
    Xtr, ytr = prep_fn(tr), tr['is_attack'].astype(int)
    Xte = prep_fn(test)[Xtr.columns]
    preds = {}
    for name, m in make_models().items():
        pipe = make_pipeline(SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True),
                             StandardScaler(), m).fit(Xtr, ytr)
        preds[name] = pipe.predict(Xte).astype(int)
    return preds

def score(yp, generator, model):
    tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
    return {'Generator': generator, 'Model': model,
            'Acc': accuracy_score(y_test, yp), 'Prec': precision_score(y_test, yp, zero_division=0),
            'Rec': recall_score(y_test, yp), 'F1': f1_score(y_test, yp),
            'Benign_FPR': fp / (fp + tn), 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

LLM_NAMES = ['Llama', 'Falcon', 'Qwen']
SYN = {k: load_syn_df(df) for k, df in syn_raw.items()}
print()
for g, s in SYN.items():
    print(f'{g:12s} rows={len(s):5d}  dups={s.duplicated().sum()}  fully-empty cols={int(s.isna().all().sum())}')
print('hard features:', prep_hard(test).shape[1])

In [ ]:
# ===== Cell 2: TSTR-hard for new sets + cross-LLM vote per setting =====
SETS = {'Real (TRTR)': attack_train, **SYN}
PREDS = {}
for g, s in SETS.items():
    print('training', g); PREDS[g] = fit_predict(s)

# Cross-LLM vote (same classifier, 3 LLMs vote, 2 of 3) -- per setting
for st in ('desc', 'B'):
    PREDS[f'XLLM_vote_{st}'] = {m: (sum(PREDS[f'{g}_{st}'][m] for g in LLM_NAMES) >= 2).astype(int)
                                for m in MODELS}

res = pd.DataFrame([score(PREDS[g][m], g, m) for g in PREDS for m in MODELS])
res.to_csv('tstr_hard_desc_B.csv', index=False)
F1 = res.pivot(index='Generator', columns='Model', values='F1')[MODELS]
F1['Mean'] = F1[MODELS].mean(axis=1)
F1['Mean_RF_LR_SVM'] = F1[['RandomForest', 'LogisticReg', 'SVM']].mean(axis=1)
FPR = res.groupby('Generator')['Benign_FPR'].mean()

# --- Anchor: TRTR must reproduce the earlier 0.994 (same pipeline) ---
print('\nTRTR anchor matches (0.994):', abs(F1.loc['Real (TRTR)', 'Mean'] - .994) < .0015)

COLS = MODELS + ['Mean', 'Mean_RF_LR_SVM']
def one(n):
    d = {c: f"{F1.loc[n, c]:.3f}" for c in COLS}; d['FPR'] = f"{FPR[n]:.3f}"; return d

rows = {'Real (TRTR)': one('Real (TRTR)')}
for g in LLM_NAMES: rows[f'{g} desc-only'] = one(f'{g}_desc')
rows['Cross-LLM vote, desc-only (2 of 3)'] = one('XLLM_vote_desc')
for g in LLM_NAMES: rows[f'{g} Setting B (attack stats)'] = one(f'{g}_B')
rows['Cross-LLM vote, Setting B (2 of 3)'] = one('XLLM_vote_B')

tab = pd.DataFrame(rows).T[COLS + ['FPR']]
tab.columns = ['RF', 'LR', 'SVM', 'MLP', 'Mean F1 (4)', 'Mean F1 (RF/LR/SVM)', 'Benign FPR']
tab.index.name = 'Training data'
print('\n===== TSTR-HARD F1: new settings =====')
print(tab.to_markdown())
print('\nReference (earlier table): CTGAN indiv 0.967 | Qwen combined 0.910 | Falcon combined 0.888 | '
      'Llama combined 0.728 | Heuristic 0.577')
tab.to_csv('tstr_table_desc_B.csv')

In [ ]:
# ===== Cell 3: Fidelity (KS, JSD, MMD²) + per-column KS spread vs real TEST attacks (HARD) =====
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics.pairwise import euclidean_distances

def ks_jsd_all(R, X, bins=20):
    ks, js = [], []
    for c in R.columns:
        r, x = R[c].values, X[c].values
        if np.ptp(np.r_[r, x]) == 0:
            ks.append(0.0); js.append(0.0); continue
        ks.append(ks_2samp(r, x).statistic)
        edges = np.unique(np.quantile(np.r_[r, x], np.linspace(0, 1, bins + 1)))
        if len(edges) < 2: edges = np.array([edges[0] - 0.5, edges[0] + 0.5])
        pr, _ = np.histogram(r, edges); px, _ = np.histogram(x, edges)
        js.append(jensenshannon(pr + 1e-12, px + 1e-12, base=2) ** 2)
    return np.mean(ks), np.mean(js)

def subs(A, n, seed): return A[np.random.default_rng(seed).choice(len(A), min(n, len(A)), replace=False)]

test_att = test[test['is_attack'] == True].drop(columns=['is_attack'])
R  = prep_hard(test_att).fillna(0).astype(float)
qt = QuantileTransformer(n_quantiles=1000, output_distribution='uniform', random_state=SEED).fit(R)
Rz = qt.transform(R)
d0 = euclidean_distances(subs(Rz, 1000, 0), squared=True)
gams = [1.0 / (np.median(d0[d0 > 0]) * f) for f in (0.1, 1, 10)]
Kf = lambda P, Q: sum(np.exp(-g * euclidean_distances(P, Q, squared=True)) for g in gams) / len(gams)

def mmd2(Xz, n=1000):
    A, B = subs(Rz, n, 1), subs(Xz, n, 2); m, k = len(A), len(B)
    Kxx, Kyy, Kxy = Kf(A, A), Kf(B, B), Kf(A, B)
    return (Kxx.sum() - np.trace(Kxx)) / (m*(m-1)) + (Kyy.sum() - np.trace(Kyy)) / (k*(k-1)) - 2 * Kxy.mean()

def ks_per_col(X):
    return pd.Series({c: 0.0 if np.ptp(np.r_[R[c].values, X[c].values]) == 0
                      else ks_2samp(R[c].values, X[c].values).statistic for c in R.columns})

# --- score each dataset ---
FSETS = {'Real-train (ref)': attack_train, **SYN}
fr, KSC = {}, {}
for g, df in FSETS.items():
    X = prep_hard(df)[R.columns].fillna(0).astype(float)
    ks, js = ks_jsd_all(R, X)
    fr[g]  = {'KS': ks, 'JSD': js, 'MMD2': mmd2(qt.transform(X))}
    KSC[g] = ks_per_col(X)
fr, KSC = pd.DataFrame(fr).T, pd.DataFrame(KSC)
fr.to_csv('fidelity_hard_desc_B.csv'); KSC.to_csv('ks_per_column_desc_B.csv')

# --- anchor: real-train must match earlier (0.0089 / 0.0010 / -0.0003) ---
print('Fidelity anchor matches:',
      max(abs(fr.loc['Real-train (ref)'].values - np.array([.0089, .0010, -.0003]))) < .0006)

# --- tables ---
const_real = [c for c in R.columns if R[c].nunique() == 1]
nonconst   = [c for c in R.columns if c not in const_real]
LABEL = {'Real-train (ref)': 'Real-train (reference)',
         **{f'{g}_desc': f'{g} desc-only' for g in LLM_NAMES},
         **{f'{g}_B': f'{g} Setting B (attack stats)' for g in LLM_NAMES}}
order = ['Real-train (ref)'] + [f'{g}_desc' for g in LLM_NAMES] + [f'{g}_B' for g in LLM_NAMES]

fid = pd.DataFrame({LABEL[n]: {'KS': f"{fr.loc[n,'KS']:.3f}", 'JSD': f"{fr.loc[n,'JSD']:.3f}",
                                'MMD²': f"{fr.loc[n,'MMD2']:.3f}"} for n in order}).T
fid.index.name = 'Training data'
print('\n===== Fidelity vs 1,346 real test attacks, HARD (lower = better) =====')
print(fid.to_markdown())
print('Reference: CTGAN indiv 0.120 / 0.107 / 0.056 | Falcon A 0.260 / 0.247 / 0.196 | '
      'Qwen A 0.336 / 0.295 / 0.227 | Llama A 0.286 / 0.271 / 0.234')

spread = pd.DataFrame({LABEL[n]: {'KS>0.5': int((KSC[n] > .5).sum()), 'KS>0.8': int((KSC[n] > .8).sum()),
                                   'KS<0.05': int((KSC[n] < .05).sum()),
                                   f'Mean KS (non-const {len(nonconst)})': f"{KSC[n][nonconst].mean():.3f}",
                                   'Max KS': f"{KSC[n].max():.3f}"} for n in order}).T
spread.index.name = 'Training data'
print(f'\n===== Per-column KS spread ({len(const_real)} of 85 cols constant in real attacks) =====')
print(spread.to_markdown())
print('Reference: CTGAN indiv KS>0.5 4.7 | non-const 0.255 || Falcon A 26.0 | 0.543 || '
      'Qwen A 33.7 | 0.643 || Llama A 28.3 | 0.580')

print('\n===== Worst 5 columns =====')
for n in order[1:]:
    w = KSC[n].sort_values(ascending=False).head(5)
    print(f'{n:12s}', ', '.join(f'{c} ({v:.2f})' for c, v in w.items()))

In [ ]:
# ===== Cell 3b: Row-level medoid (desc-only + Setting B) + TSTR + fidelity =====
# Same 27 distance columns as the Setting A medoid (core LLM columns, no IDs/proto/context)
DIST_COLS = ['pkts_fwd','bytes_fwd','duration','pkt_size_std','pkt_size_min','pkt_size_max',
             'mean_hop_count','mean_hop_latency','mean_iat_int_ns','iat_std_int_ns','iat_min_int_ns',
             'iat_max_int_ns','latency_min','latency_max','latency_std','syn_count_fwd','ack_count_fwd',
             'fin_count_fwd','rst_count_fwd','psh_count_fwd','urg_count_fwd','small_pkt_fraction_total',
             'tiny_pkt_fraction_total','queue_occupancy_mean','queue_occupancy_std','queue_occupancy_max',
             'truncated_udp_fraction']

pick_counts = {}
for st in ('desc', 'B'):
    srcs = [SYN[f'{g}_{st}'].reset_index(drop=True) for g in LLM_NAMES]
    tf = [syn_raw[f'{g}_{st}']['timeframe_id'].astype(str).values for g in LLM_NAMES]
    print(f'{st}: rows aligned by timeframe = {((tf[0] == tf[1]) & (tf[1] == tf[2])).mean():.1%}')

    Xs = [s[DIST_COLS].apply(pd.to_numeric, errors='coerce').fillna(0).values for s in srcs]
    qt_s = QuantileTransformer(n_quantiles=1000, random_state=SEED).fit(np.vstack(Xs))   # synthetic only
    L, F, Q = (qt_s.transform(x) for x in Xs)
    dLF, dLQ, dFQ = (np.linalg.norm(a - b, axis=1) for a, b in [(L, F), (L, Q), (F, Q)])
    tot = np.stack([dLF + dLQ, dLF + dFQ, dLQ + dFQ], 1)          # 0=Llama 1=Falcon 2=Qwen
    is_min = np.isclose(tot, tot.min(1, keepdims=True)); ties = is_min.sum(1) > 1
    pick = np.argmin(tot, axis=1); pick[ties & is_min[:, 1]] = 1  # tie involving Falcon -> Falcon
    print(f'{st}: ties {int(ties.sum())}')
    SYN[f'Medoid_{st}'] = pd.concat([srcs[k].iloc[[i]] for i, k in enumerate(pick)], ignore_index=True)
    pick_counts[st] = {g: int((pick == k).sum()) for k, g in enumerate(LLM_NAMES)}
print('\n=== Which LLM row was kept (per 3,300) ===\n', pd.DataFrame(pick_counts).to_string())

# --- TSTR + fidelity (same pipeline, same test set) ---
MED = ['Medoid_desc', 'Medoid_B']
for g in MED:
    print('training', g); PREDS[g] = fit_predict(SYN[g])
    X = prep_hard(SYN[g])[R.columns].fillna(0).astype(float)
    ks, js = ks_jsd_all(R, X)
    fr.loc[g] = {'KS': ks, 'JSD': js, 'MMD2': mmd2(qt.transform(X))}
    KSC[g] = ks_per_col(X)

res = pd.concat([res[~res['Generator'].isin(MED)],
                 pd.DataFrame([score(PREDS[g][m], g, m) for g in MED for m in MODELS])], ignore_index=True)
F1 = res.pivot(index='Generator', columns='Model', values='F1')[MODELS]
F1['Mean'] = F1[MODELS].mean(axis=1)
F1['Mean_RF_LR_SVM'] = F1[['RandomForest', 'LogisticReg', 'SVM']].mean(axis=1)
FPR = res.groupby('Generator')['Benign_FPR'].mean()

med = pd.DataFrame({{'Medoid_desc': 'Medoid desc-only', 'Medoid_B': 'Medoid Setting B'}[n]:
    {**{c: f"{F1.loc[n, c]:.3f}" for c in COLS}, 'FPR': f"{FPR[n]:.3f}",
     'KS': f"{fr.loc[n,'KS']:.3f}", 'JSD': f"{fr.loc[n,'JSD']:.3f}", 'MMD²': f"{fr.loc[n,'MMD2']:.3f}"}
    for n in MED}).T
med.columns = ['RF', 'LR', 'SVM', 'MLP', 'Mean F1 (4)', 'Mean F1 (RF/LR/SVM)', 'Benign FPR', 'KS', 'JSD', 'MMD²']
print('\n', med.to_markdown())

In [ ]:
# ===== Cell 5: Vote + Medoid + Fusion for desc-only and Setting B =====
import json, glob
from sklearn.preprocessing import QuantileTransformer

# ---------- rules: benign for desc-only, attack for Setting B ----------
def load_json(*names):
    for n in names:
        hits = sorted(glob.glob(f'/content/**/{glob.escape(n)}', recursive=True))
        if hits: return json.load(open(hits[0])), hits[0]
    raise FileNotFoundError(names)
rules,   f1 = load_json('rules.json', 'rules (1).json', 'rules(1).json')   # desc-only (benign scaffold)
rules_B, f2 = load_json('rules_attack.json')                               # Setting B (attack, train split)
print('desc rules:', f1, '| B rules:', f2)
assert rules['derived'] == rules_B['derived'], 'derived formulas differ!'

ID_FUSE, NOMINAL = ['flow_key', 'timeframe_id', 'ip_a', 'ip_b', 'port_a', 'port_b'], ['proto']
ROLE = {}
for c in train.columns.drop('is_attack'):
    if c in ID_FUSE:                      ROLE[c] = 'id'
    elif c in rules['derived']:           ROLE[c] = 'derived'
    elif c in rules['pinned']:            ROLE[c] = 'pinned'
    elif c in rules['always_empty']:      ROLE[c] = 'empty'
    elif c in NOMINAL:                    ROLE[c] = 'nominal'
    elif rules['dtypes'].get(c) == 'int': ROLE[c] = 'integer'
    else:                                 ROLE[c] = 'continuous'
FUSE_NUM = [c for c, t in ROLE.items() if t in ('integer', 'continuous')]

# same 27 distance columns as the Setting A medoid
DIST_COLS = ['pkts_fwd','bytes_fwd','duration','pkt_size_std','pkt_size_min','pkt_size_max',
             'mean_hop_count','mean_hop_latency','mean_iat_int_ns','iat_std_int_ns','iat_min_int_ns',
             'iat_max_int_ns','latency_min','latency_max','latency_std','syn_count_fwd','ack_count_fwd',
             'fin_count_fwd','rst_count_fwd','psh_count_fwd','urg_count_fwd','small_pkt_fraction_total',
             'tiny_pkt_fraction_total','queue_occupancy_mean','queue_occupancy_std','queue_occupancy_max',
             'truncated_udp_fraction']

# ---------- fusion rules (identical to Setting A) ----------
def fuse_numeric(V, integer=False):
    out = np.full(len(V), np.nan); n_nan = np.isnan(V).sum(1)
    m1 = n_nan == 1; out[m1] = np.nanmean(V[m1], axis=1)
    m0 = n_nan == 0; A = np.sort(V[m0], axis=1); g1, g2 = A[:, 1] - A[:, 0], A[:, 2] - A[:, 1]
    out[m0] = np.where(g1 < g2, (A[:, 0] + A[:, 1]) / 2, np.where(g2 < g1, (A[:, 1] + A[:, 2]) / 2, A[:, 1]))
    return np.floor(out) if integer else out

def fuse_nominal(V, rng):
    out, n = [], 0
    for a, b, c in V:
        if a == b or a == c: out.append(a)
        elif b == c:         out.append(b)
        else:                out.append([a, b, c][rng.integers(3)]); n += 1
    return np.array(out), n

def recompute_derived(d):
    d = d.copy()
    for col, f in rules['derived'].items():
        if col in d.columns:
            d[col] = pd.Series(d.eval(f, engine='python'), index=d.index).astype(float).replace([np.inf, -np.inf], np.nan)
    return d

def apply_empty_rules(d):
    for er in rules['empty_rules']:
        for cond in er['must_be_empty_if']:
            try: m = d.eval(cond, engine='python').astype(bool)
            except Exception: continue
            d.loc[m, [c for c in er['columns'] if c in d.columns]] = np.nan
    return d

def fuse(srcs, setting, seed=42):
    rng, rep = np.random.default_rng(seed), {}
    out = srcs[1].copy()                                    # IDs copied (dropped by ML)
    out['proto'], rep['proto all differ'] = fuse_nominal(np.stack([s['proto'].values for s in srcs], 1), rng)
    for c in FUSE_NUM:
        V = np.stack([pd.to_numeric(s[c], errors='coerce').values.astype(float) for s in srcs], 1)
        if c == 'duration': Vdur = V
        out[c] = fuse_numeric(V, integer=(ROLE[c] == 'integer'))
    for c, v in (rules_B if setting == 'B' else rules)['pinned'].items():
        if c in out.columns and c not in ID_FUSE: out[c] = v
    for c in rules['always_empty']: out[c] = np.nan
    if setting == 'B': out['ack_count_fwd'] = out['pkts_fwd']           # B prompt formula
    p1 = out['pkts_fwd'] == 1; out.loc[p1, 'duration'] = 0.0
    bad = ((out['pkts_fwd'] > 1) & (out['duration'] == 0)).values
    alt = np.nanmean(np.where(Vdur > 0, Vdur, np.nan), axis=1)
    fix = bad & ~np.isnan(alt); out.loc[fix, 'duration'] = alt[fix]
    n_clip = 0
    for f in ('syn', 'ack', 'fin', 'rst', 'psh', 'urg'):
        c = f'{f}_count_fwd'; over = out[c] > out['pkts_fwd']; n_clip += int(over.sum())
        out.loc[over, c] = out.loc[over, 'pkts_fwd']
    rep['flag counts clipped'] = n_clip
    out = recompute_derived(out)
    for lo, mid, hi in [('pkt_size_min', 'mean_pkt_size', 'pkt_size_max'),
                        ('iat_min_int_ns', 'mean_iat_int_ns', 'iat_max_int_ns'),
                        ('latency_min', 'mean_hop_latency', 'latency_max')]:
        rep[f'{mid} range fixed'] = int(((out[lo] > out[mid]) | (out[mid] > out[hi])).sum())
        out[lo], out[hi] = np.fmin(out[lo], out[mid]), np.fmax(out[hi], out[mid])
    out['pkt_size_min'], out['pkt_size_max'] = np.floor(out['pkt_size_min']), np.ceil(out['pkt_size_max'])
    return apply_empty_rules(out), rep

# ---------- alignment ----------
def aligned_triplets(st):
    if st == 'desc':                                         # same row order (100% aligned)
        return [SYN[f'{g}_desc'].reset_index(drop=True) for g in LLM_NAMES], 3300
    srcs = {g: SYN[f'{g}_B'].reset_index(drop=True).assign(  # B: align by slot (timeframe x single/multi)
                _tf=syn_raw[f'{g}_B']['timeframe_id'].astype(int).values,
                _single=(SYN[f'{g}_B']['pkts_fwd'] == 1).values) for g in LLM_NAMES}
    grp = pd.DataFrame({g: s.groupby(['_tf', '_single']).size() for g, s in srcs.items()}).fillna(0).astype(int)
    n_pair = grp.min(axis=1)
    out = {g: [] for g in LLM_NAMES}
    for (tf, sg), n in n_pair.items():
        for g in LLM_NAMES:
            out[g].append(srcs[g][(srcs[g]['_tf'] == tf) & (srcs[g]['_single'] == sg)].head(n))
    return [pd.concat(out[g], ignore_index=True).drop(columns=['_tf', '_single']) for g in LLM_NAMES], int(n_pair.sum())

def medoid(srcs):
    Xs = [s[DIST_COLS].apply(pd.to_numeric, errors='coerce').fillna(0).values for s in srcs]
    qt_s = QuantileTransformer(n_quantiles=1000, random_state=SEED).fit(np.vstack(Xs))   # synthetic only
    L, F, Q = (qt_s.transform(x) for x in Xs)
    dLF, dLQ, dFQ = (np.linalg.norm(a - b, axis=1) for a, b in [(L, F), (L, Q), (F, Q)])
    tot = np.stack([dLF + dLQ, dLF + dFQ, dLQ + dFQ], 1)          # 0=Llama 1=Falcon 2=Qwen
    is_min = np.isclose(tot, tot.min(1, keepdims=True)); ties = is_min.sum(1) > 1
    pick = np.argmin(tot, axis=1); pick[ties & is_min[:, 1]] = 1  # tie involving Falcon -> Falcon
    m = pd.concat([srcs[k].iloc[[i]] for i, k in enumerate(pick)], ignore_index=True)
    return m, int(ties.sum()), {g: int((pick == k).sum()) for k, g in enumerate(LLM_NAMES)}

# ---------- build medoid + fused for both settings ----------
NEWSETS = []
for st in ('desc', 'B'):
    srcs, n_rows = aligned_triplets(st)
    SYN[f'Medoid_{st}'], ties, kept = medoid(srcs)
    SYN[f'Fused_{st}'], rep = fuse(srcs, st)
    print(f'\n[{st}] aligned rows: {n_rows} | medoid ties {ties} | kept {kept}\n  fusion report: {rep}')
    NEWSETS += [f'Medoid_{st}', f'Fused_{st}']

# ---------- TSTR + fidelity + KS ----------
for g in NEWSETS:
    print('training', g); PREDS[g] = fit_predict(SYN[g])
    X = prep_hard(SYN[g])[R.columns].fillna(0).astype(float)
    ks, js = ks_jsd_all(R, X)
    fr.loc[g] = {'KS': ks, 'JSD': js, 'MMD2': mmd2(qt.transform(X))}
    KSC[g] = ks_per_col(X)
res = pd.concat([res[~res['Generator'].isin(NEWSETS)],
                 pd.DataFrame([score(PREDS[g][m], g, m) for g in NEWSETS for m in MODELS])], ignore_index=True)
F1 = res.pivot(index='Generator', columns='Model', values='F1')[MODELS]
F1['Mean'] = F1[MODELS].mean(axis=1)
F1['Mean_RF_LR_SVM'] = F1[['RandomForest', 'LogisticReg', 'SVM']].mean(axis=1)
FPR = res.groupby('Generator')['Benign_FPR'].mean()

# ---------- tables ----------
COLS = MODELS + ['Mean', 'Mean_RF_LR_SVM']
def row(n, fid=True):
    d = {c: f"{F1.loc[n, c]:.3f}" for c in COLS}; d['FPR'] = f"{FPR[n]:.3f}"
    for c, k in [('KS', 'KS'), ('JSD', 'JSD'), ('MMD²', 'MMD2')]:
        d[c] = f"{fr.loc[n, k]:.3f}" if fid else '—'
    d['KS>0.5'] = int((KSC[n] > .5).sum()) if fid else '—'
    d['Mean KS non-const'] = f"{KSC[n][nonconst].mean():.3f}" if fid else '—'
    return d

HDR = ['RF', 'LR', 'SVM', 'MLP', 'Mean F1 (4)', 'Mean F1 (RF/LR/SVM)', 'Benign FPR',
       'KS', 'JSD', 'MMD²', 'KS>0.5', 'Mean KS non-const']
for st, name in [('desc', 'DESCRIPTION-ONLY'), ('B', 'SETTING B (attack stats)')]:
    rows = {g: row(f'{g}_{st}') for g in LLM_NAMES}
    rows['Cross-LLM vote (2 of 3)'] = row(f'XLLM_vote_{st}', fid=False)
    rows['Medoid'] = row(f'Medoid_{st}')
    rows['Fused (Hadi)'] = row(f'Fused_{st}')
    t = pd.DataFrame(rows).T[COLS + ['FPR', 'KS', 'JSD', 'MMD²', 'KS>0.5', 'Mean KS non-const']]
    t.columns = HDR; t.index.name = name
    print(f'\n===== {name} =====\n'); print(t.to_markdown())
    t.to_csv(f'combine_{st}.csv')
print('\nReference: CTGAN 0.967 | MMD² 0.056 | KS>0.5 4.7 | non-const 0.255')

In [ ]:
# ===== Cell 7: Where does each classifier miss? (single- vs multi-packet real attacks) =====
tp_pkts = pd.to_numeric(test['pkts_fwd'], errors='coerce').values
att = y_test == 1
groups = {'single-pkt attacks': att & (tp_pkts == 1),
          'multi-pkt attacks':  att & (tp_pkts > 1),
          'benign':             ~att}
print({k: int(v.sum()) for k, v in groups.items()})

rows = []
for g in ['Falcon_B', 'Llama_B', 'Qwen_B', 'Falcon_desc', 'Qwen_desc']:
    syn_single = (pd.to_numeric(SYN[g]['pkts_fwd'], errors='coerce') == 1).mean()
    for m in MODELS:
        p = PREDS[g][m]
        rows.append({'Generator': g, 'synthetic single-pkt %': f'{syn_single:.1%}', 'Model': m,
                     'recall single-pkt': p[groups['single-pkt attacks']].mean(),
                     'recall multi-pkt':  p[groups['multi-pkt attacks']].mean(),
                     'FPR benign':        p[groups['benign']].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
# ===== Cell 8: Single-packet flows - synthetic vs real attack vs real benign (medians) =====
FEATS = ['bytes_fwd', 'mean_pkt_size', 'pkt_size_min', 'pkt_size_max', 'psh_count_fwd', 'ack_count_fwd',
         'syn_count_fwd', 'small_pkt_fraction_total', 'mean_hop_latency', 'latency_min', 'latency_max',
         'latency_std', 'queue_occupancy_mean', 'queue_occupancy_max', 'queue_occupancy_std', 'mean_hop_count']
def single(d): return d[pd.to_numeric(d['pkts_fwd'], errors='coerce') == 1]

real_att_1 = single(load_syn_df(test[test['is_attack'] == True]))
real_ben_1 = single(load_syn_df(pd.concat([benign_train, test[test['is_attack'] == False]])))
cols = {'REAL attack (1-pkt)': real_att_1, 'REAL benign (1-pkt)': real_ben_1,
        **{f'{g} B (1-pkt)': single(SYN[f'{g}_B']) for g in LLM_NAMES}}
print({k: len(v) for k, v in cols.items()})
tab = pd.DataFrame({k: v[FEATS].median() for k, v in cols.items()})
tab.loc['INT empty %'] = [v['mean_hop_latency'].isna().mean() * 100 for v in cols.values()]
print(tab.to_string(float_format=lambda x: f'{x:,.4g}'))

In [ ]:
# ===== Final results tables -> Excel (Detection + Fidelity) for professor / Hadi =====
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

S = None   # section marker

DET_COLS = ['RF', 'LR', 'SVM', 'MLP', 'Mean F1 (4)', 'Mean F1 (RF/LR/SVM)', 'Benign FPR']
det = [
 ('Real (TRTR)',          ['0.995','0.994','0.993','0.994','0.994','0.994','0.014']),
 ('Heuristic',            ['0.404','0.910','0.087','0.909','0.577','0.467','0.015']),
 ('CTGAN (3 seeds)',      ['0.910 ± 0.000','0.984 ± 0.002','0.981 ± 0.002','0.993 ± 0.001','0.967 ± 0.001','0.958 ± 0.001','0.004']),
 ('CTGAN combined',       ['0.911','0.984','0.983','0.994','0.968','0.960','0.004']),
 ('Setting A: benign stats', S),
 ('Llama (3 runs)',       ['0.906 ± 0.000','0.596 ± 0.147','0.717 ± 0.095','0.634 ± 0.082','0.713 ± 0.080','0.740 ± 0.080','0.009']),
 ('Falcon (3 runs)',      ['0.910 ± 0.000','0.903 ± 0.008','0.906 ± 0.000','0.535 ± 0.166','0.813 ± 0.043','0.906 ± 0.003','0.015']),
 ('Qwen (3 runs)',        ['0.909 ± 0.002','0.893 ± 0.004','0.903 ± 0.001','0.911 ± 0.046','0.904 ± 0.012','0.902 ± 0.002','0.012']),
 ('Llama combined',       ['0.907','0.610','0.822','0.571','0.728','0.780','0.010']),
 ('Falcon combined',      ['0.912','0.907','0.906','0.828','0.888','0.908','0.021']),
 ('Qwen combined',        ['0.911','0.899','0.904','0.926','0.910','0.905','0.012']),
 ('Cross-LLM vote (a)',   ['0.911','0.900','0.904','0.822','0.884','0.905','0.009']),
 ('Medoid (3 runs)',      ['0.910 ± 0.000','0.853 ± 0.003','0.895 ± 0.007','0.796 ± 0.071','0.864 ± 0.019','0.886 ± 0.002','0.014']),
 ('Fused (3 runs)',       ['0.908 ± 0.001','0.893 ± 0.006','0.868 ± 0.005','0.877 ± 0.001','0.886 ± 0.003','0.890 ± 0.004','0.013']),
 ('Setting B: attack stats', S),
 ('Llama ',               ['0.904','0.894','0.892','0.907','0.899','0.897','0.010']),
 ('Falcon ',              ['0.994','0.861','0.871','0.917','0.911','0.909','0.003']),
 ('Qwen ',                ['0.907','0.906','0.903','0.908','0.906','0.905','0.014']),
 ('Cross-LLM vote ',      ['0.908','0.902','0.895','0.908','0.903','0.902','0.009']),
 ('Medoid (b)',           ['0.907','0.905','0.901','0.907','0.905','0.905','0.015']),
 ('Fused (b)',            ['0.907','0.904','0.902','0.905','0.905','0.904','0.012']),
 ('Description-only: no data', S),
 ('Llama  ',              ['0.902','0.000','0.013','0.000','0.229','0.305','0.001']),
 ('Falcon  ',             ['0.909','0.704','0.895','0.472','0.745','0.836','0.019']),
 ('Qwen  ',               ['0.906','0.894','0.901','0.911','0.903','0.901','0.000']),
 ('Cross-LLM vote  ',     ['0.906','0.693','0.890','0.269','0.689','0.830','0.000']),
 ('Medoid  ',             ['0.907','0.087','0.259','0.131','0.346','0.417','0.017']),
 ('Fused  ',              ['0.904','0.878','0.850','0.811','0.861','0.878','0.027']),
]

FID_COLS = ['KS', 'JSD', 'MMD²', 'KS > 0.5 (of 85)', 'Mean KS (40 non-const)']
fid = [
 ('Real-train (reference)', ['0.009','0.001','0.000','0','0.019']),
 ('Heuristic (c)',          ['0.285','n/a','0.288','n/a','n/a']),
 ('CTGAN (3 seeds)',        ['0.120 ± 0.005','0.107 ± 0.001','0.056 ± 0.004','4.7 ± 1.5','0.255 ± 0.010']),
 ('CTGAN combined',         ['0.105','0.094','0.043','3','0.224']),
 ('Setting A: benign stats', S),
 ('Llama (3 runs)',         ['0.286 ± 0.000','0.271 ± 0.003','0.234 ± 0.003','28.3 ± 0.6','0.580 ± 0.001']),
 ('Falcon (3 runs)',        ['0.260 ± 0.001','0.247 ± 0.006','0.196 ± 0.001','26.0 ± 1.7','0.543 ± 0.002']),
 ('Qwen (3 runs)',          ['0.336 ± 0.001','0.295 ± 0.002','0.227 ± 0.002','33.7 ± 0.6','0.643 ± 0.000']),
 ('Llama combined',         ['0.286','0.259','0.230','28','0.580']),
 ('Falcon combined',        ['0.260','0.235','0.196','27','0.543']),
 ('Qwen combined',          ['0.336','0.288','0.239','34','0.643']),
 ('Cross-LLM vote',         ['—','—','—','—','—']),
 ('Medoid (3 runs)',        ['0.282 ± 0.000','0.258 ± 0.001','0.204 ± 0.002','29.0 ± 0.0','0.573 ± 0.000']),
 ('Fused (3 runs)',         ['0.278 ± 0.000','0.273 ± 0.001','0.228 ± 0.002','26.0 ± 0.0','0.572 ± 0.001']),
 ('Setting B: attack stats', S),
 ('Llama ',                 ['0.143','0.142','0.087','4','0.285']),
 ('Falcon ',                ['0.132','0.127','0.086','3','0.276']),
 ('Qwen ',                  ['0.175','0.162','0.083','8','0.344']),
 ('Cross-LLM vote ',        ['—','—','—','—','—']),
 ('Medoid (b)',             ['0.146','0.144','0.065','2','0.293']),
 ('Fused (b)',              ['0.122','0.129','0.038','1','0.259']),
 ('Description-only: no data', S),
 ('Llama  ',                ['0.291','0.285','0.163','23','0.553']),
 ('Falcon  ',               ['0.275','0.255','0.166','27','0.544']),
 ('Qwen  ',                 ['0.345','0.326','0.206','37','0.617']),
 ('Cross-LLM vote  ',       ['—','—','—','—','—']),
 ('Medoid  ',               ['0.285','0.276','0.165','26','0.548']),
 ('Fused  ',                ['0.276','0.286','0.187','28','0.563']),
]

# cells that beat CTGAN (3 seeds) -> highlighted
BEAT = {('Detection', 'Falcon ', 'RF'), ('Detection', 'Falcon ', 'Benign FPR'),
        ('Detection', 'Qwen  ', 'Benign FPR'), ('Detection', 'Cross-LLM vote  ', 'Benign FPR'),
        ('Fidelity', 'Fused (b)', 'MMD²'), ('Fidelity', 'Fused (b)', 'KS > 0.5 (of 85)'),
        ('Fidelity', 'Medoid (b)', 'KS > 0.5 (of 85)'), ('Fidelity', 'Falcon ', 'KS > 0.5 (of 85)'),
        ('Fidelity', 'Llama ', 'KS > 0.5 (of 85)')}

NOTES = [
 'TSTR-hard: 85 per-flow features; each detector trained on 3,300 synthetic attacks + the same 404 real benign; '
 'tested on the same 1,540 real flows (1,346 attacks, 194 benign). Fidelity vs the 1,346 real test attacks (lower = better).',
 'Highlighted = LLM value better than CTGAN (3 seeds). Settings B and description-only are single runs (no combined rows).',
 '(a) Setting A vote uses the combined models (runs 1-3 merged per LLM). Medoid/Fused shown per run; '
 'their combined versions: Medoid 0.901 / MMD² 0.203, Fused 0.894 / MMD² 0.228.',
 '(b) Setting B medoid/fused built on 3,141 slot-aligned rows (timeframe x single/multi-packet), as B files differ in row order.',
 '(c) Heuristic KS/MMD² from the earlier run with the same method; JSD and KS spread not computed.',
 'Setting B uses attack statistics mined from the same 3,116 training attacks CTGAN is trained on (summary stats only, no rows).',
]

def to_df(rows, cols):
    idx = [r[0] for r in rows]
    data = [r[1] if r[1] is not None else [''] * len(cols) for r in rows]
    return pd.DataFrame(data, index=idx, columns=cols).rename_axis('Training data')

DET, FID = to_df(det, DET_COLS), to_df(fid, FID_COLS)
print('TABLE 1 - DETECTION (TSTR-hard F1)\n'); print(DET.to_markdown())
print('\nTABLE 2 - FIDELITY (lower = better)\n'); print(FID.to_markdown())

OUT = 'ICC_final_results.xlsx'
with pd.ExcelWriter(OUT, engine='openpyxl') as xw:
    for name, df, rows in [('Detection', DET, det), ('Fidelity', FID, fid)]:
        df.to_excel(xw, sheet_name=name, startrow=1)
        ws = xw.sheets[name]
        ws['A1'] = ('Table 1 - Detection utility (TSTR-hard F1)' if name == 'Detection'
                    else 'Table 2 - Fidelity vs real test attacks (lower = better)')
        ws['A1'].font = Font(bold=True, size=13)
        hdr = PatternFill('solid', fgColor='D9E1F2'); sec = PatternFill('solid', fgColor='EDEDED')
        win = PatternFill('solid', fgColor='C6EFCE')
        for c in ws[2]: c.font = Font(bold=True); c.fill = hdr; c.alignment = Alignment(horizontal='center', wrap_text=True)
        cols = list(df.columns)
        for i, (label, vals) in enumerate(rows, start=3):
            if vals is None:
                for c in ws[i]: c.fill = sec
                ws.cell(i, 1).font = Font(bold=True, italic=True)
                continue
            for j, col in enumerate(cols, start=2):
                cell = ws.cell(i, j); cell.alignment = Alignment(horizontal='center')
                if (name, label, col) in BEAT: cell.fill = win; cell.font = Font(bold=True)
            if label.strip().startswith(('Real', 'CTGAN')): ws.cell(i, 1).font = Font(bold=True)
        r = len(rows) + 4
        for n in NOTES:
            ws.cell(r, 1, n).font = Font(italic=True, size=9); r += 1
        ws.column_dimensions['A'].width = 30
        for j in range(2, len(cols) + 2): ws.column_dimensions[get_column_letter(j)].width = 17
        ws.freeze_panes = 'B3'

from google.colab import files
files.download(OUT)